# scVI-LD Baseline for Single-Cell RNA-seq Analysis

This notebook provides a baseline implementation using [scVI's Latent Dirichlet (LD) model] for interpretable single-cell RNA-seq data analysis.

## 📋 Prerequisites

Before running this baseline, please ensure you have:

1. **Followed the main installation guide** in the root directory's README to set up the conda environment
2. **Installed additional dependencies** specific to scVI (same as the scVI baseline):

```bash
# Activate the scE2TM environment first
conda activate scE2TM_env

# Install scVI and its dependencies (with torch version locked to match your environment)
pip install scvi-tools==0.14.0 torch==1.9.1+cu102 -f https://download.pytorch.org/whl/torch_stable.html

# Install torchmetrics without dependencies (to avoid version conflicts)
pip install torchmetrics==0.5.0 --no-deps

pip install setuptools==59.5.0

In [ ]:
"""
scVI-LD (Linear Decoder) baseline implementation for single-cell RNA-seq analysis
This script performs dimensionality reduction, clustering, and evaluation using scVI's LinearSCVI model
which provides more interpretable latent representations through linear decoding
"""

import scvi
import scanpy as sc
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, silhouette_score
from sklearn import metrics
import os
import sys
import time

# Get the current directory - works in both Jupyter notebook and Python scripts
try:
    # This works when running as a script
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # This works in Jupyter notebook
    current_dir = os.getcwd()

print(f"Current working directory: {current_dir}")

# Define data paths (relative to the script location)
data_name = 'Wang'  # Base name for the data files
data_path = os.path.join(current_dir, '..', 'data', f'{data_name}_HIGHPRE.csv')  # Expression data
label_path = os.path.join(current_dir, '..', 'data', f'{data_name}_cell_anno.csv')  # Cell type annotations

# Define output directory (create if it doesn't exist)
output_dir = os.path.join(current_dir, 'output')
os.makedirs(output_dir, exist_ok=True)

print(f"Loading data from: {data_path}")
print(f"Loading labels from: {label_path}")
print(f"Outputs will be saved to: {output_dir}")

# Check if files exist
if not os.path.exists(data_path):
    print(f"ERROR: Data file not found at {data_path}")
    print("Please make sure the data file is in the correct location.")
    print("Expected structure:")
    print("  scE2TM/")
    print("  ├── baseline/")
    print("  │   ├── scVI_LD.ipynb (this notebook)")
    print("  │   └── output/ (will be created)")
    print("  └── data/")
    print("      ├── Wang_HIGHPRE.csv")
    print("      └── Wang_cell_anno.csv")
    sys.exit(1)

if not os.path.exists(label_path):
    print(f"ERROR: Label file not found at {label_path}")
    print("Please make sure the label file is in the correct location.")
    sys.exit(1)

# Load expression data
# The data should be in CSV format with genes as columns and cells as rows
scvi_csv = pd.read_csv(data_path, sep=',', index_col=0)
print(f"Data shape: {scvi_csv.shape}")

# Create AnnData object (standard format for single-cell data)
scvild_adata = sc.AnnData(scvi_csv)

# Start timing
start_time = time.time()

# Setup LinearSCVI model
# This prepares the AnnData object for training
print("Setting up LinearSCVI model...")
scvi.model.LinearSCVI.setup_anndata(scvild_adata)

# Initialize and train LinearSCVI model
# n_latent: dimension of the latent space (we use 100 for better representation)
# LinearSCVI uses a linear decoder which provides more interpretable latent factors
model = scvi.model.LinearSCVI(scvild_adata, n_latent=100)
print("Training LinearSCVI model...")
model.train(use_gpu=0)  # Use GPU 0 for training

# Extract latent representation (low-dimensional embedding)
latent = model.get_latent_representation()
scvild_adata.obsm["X_scVI_LD"] = latent
print(f"Latent representation shape: {latent.shape}")

# Extract topic-gene loading matrix
# This matrix represents the relationship between topics (latent factors) and genes
# Each row corresponds to a topic, each column to a gene
print("Extracting topic-gene loading matrix...")
topic_gene = model.get_loadings()  # Shape: (n_topics, n_genes)
print(f"Topic-gene matrix shape: {topic_gene.shape}")

# End timing
end_time = time.time()
print(f"Training time: {end_time - start_time:.2f} seconds")

# Load cell type labels
# These are the ground truth labels for evaluation
scvi_label_csv = pd.read_csv(label_path, sep=',', index_col=0)
scvild_adata.obs['cell_type'] = list(scvi_label_csv.iloc[:, 0])
print(f"Number of cell types: {len(scvild_adata.obs['cell_type'].unique())}")

# Handle potential NaN values in labels
non_nan_indices = [i for i, x in enumerate(scvild_adata.obs['cell_type']) if pd.notna(x)]
print(f"Cells with valid labels: {len(non_nan_indices)}/{len(scvild_adata.obs['cell_type'])}")

# Construct neighborhood graph using scVI-LD latent representation
sc.pp.neighbors(scvild_adata, n_neighbors=15, use_rep="X_scVI_LD")

# Find optimal resolution for Louvain clustering
# We search over a range of resolutions to maximize ARI
max_resolution = 2  # Maximum resolution to test
min_resolution = 0  # Minimum resolution to test
n_steps = 20  # Number of resolution steps (max_resolution * 10)
ari_scores = []

print("Searching for optimal clustering resolution...")
for i in range(min_resolution, n_steps):
    resolution = i / 10.0
    sc.tl.louvain(scvild_adata, resolution=resolution, random_state=0)
    # Only use cells with valid labels for ARI calculation
    ari = adjusted_rand_score(
        scvild_adata.obs['cell_type'][non_nan_indices], 
        scvild_adata.obs['louvain'][non_nan_indices]
    )
    ari_scores.append(ari)
    print(f"Resolution {resolution:.1f}: ARI = {ari:.4f}")

# Select the resolution that gives the highest ARI
best_resolution = ari_scores.index(max(ari_scores)) * 0.1
print(f"Best resolution: {best_resolution:.1f} (ARI = {max(ari_scores):.4f})")

# Perform final clustering with optimal resolution
sc.tl.louvain(scvild_adata, resolution=best_resolution, random_state=0)

# Compute UMAP for visualization
sc.tl.umap(scvild_adata)

# Visualize results
sc.pl.umap(
    scvild_adata,
    color=["cell_type", "louvain"],
    wspace=0.3,
    title=['Ground Truth Cell Types', 'scVI-LD Clustering Results']
    # Uncomment the line below to save the figure
    # save=os.path.join(output_dir, 'scVI_LD_umap.pdf')
)

# Calculate and print evaluation metrics
ari = adjusted_rand_score(
    scvild_adata.obs['cell_type'][non_nan_indices], 
    scvild_adata.obs['louvain'][non_nan_indices]
)
ami = adjusted_mutual_info_score(
    scvild_adata.obs['cell_type'][non_nan_indices], 
    scvild_adata.obs['louvain'][non_nan_indices]
)
asw = silhouette_score(
    scvild_adata.obsm["X_scVI_LD"][non_nan_indices], 
    scvild_adata.obs['cell_type'][non_nan_indices]
)

print("\n" + "="*50)
print("EVALUATION METRICS")
print("="*50)
print(f"Adjusted Rand Index (ARI):      {ari:.4f}")
print(f"Adjusted Mutual Information (AMI): {ami:.4f}")
print(f"Average Silhouette Width (ASW): {asw:.4f}")
print("="*50)

# Calculate purity score based on dominant topic
# For LinearSCVI, the latent dimensions can be interpreted as topic weights
# The dominant topic is the dimension with the highest value
dominant_topics = np.argmax(scvild_adata.obsm["X_scVI_LD"], axis=1)

def purity_score(y_true, y_pred):
    """
    Calculate purity score for clustering evaluation
    Purity measures the extent to which each cluster contains only cells from a single class
    """
    contingency_matrix = metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)

purity = purity_score(scvild_adata.obs['cell_type'][non_nan_indices], dominant_topics[non_nan_indices])
print(f"Purity Score (based on dominant topic): {purity:.4f}")
print("="*50)

# ============================================================
# SAVE OUTPUTS
# ============================================================
print("\n" + "="*50)
print("SAVING OUTPUTS")
print("="*50)
print(f"All outputs will be saved to: {output_dir}")

# Save topic-gene matrix
topic_gene_path = os.path.join(output_dir, f'SCVILD_{data_name}_topic_gene.csv')
pd.DataFrame(topic_gene, 
             index=[f'Topic_{i+1}' for i in range(topic_gene.shape[0])],
             columns=scvild_adata.var_names).to_csv(topic_gene_path)
print(f"✓ Topic-gene matrix saved: SCVILD_{data_name}_topic_gene.csv")

# Save cell embeddings
embedding_path = os.path.join(output_dir, f'SCVILD_{data_name}_embedding.csv')
pd.DataFrame(latent,
             index=scvild_adata.obs_names,
             columns=[f'Dim_{i+1}' for i in range(latent.shape[1])]).to_csv(embedding_path)
print(f"✓ Cell embeddings saved: SCVILD_{data_name}_embedding.csv")

print("="*50)
print("DONE!")
print("="*50)